# 09 — Ajuste fino com o piloto Open Images e validação Home

Execute em uma **sessão nova do Colab, com GPU T4**. As 5.000 imagens já preparadas pelo
08 serão usadas junto às **1.824 imagens Home TRAIN**, preservando a validação Home de
456 imagens. Iniciamos no modelo antigo; os ajustes SUN anteriores continuam registrados.

| Controle | Definição deste experimento |
|---|---|
| Pesos iniciais | Antigo `modelos/yolov8n_baseline/best.pt` |
| Imagens únicas de treino | 1.824 Home + 5.000 Open Images = 6.824 |
| Apresentações por época | 5.000 Open Images + 5.000 Home reamostradas = 10.000 |
| Batch | 16: metade Home e metade Open Images |
| Épocas | Até 15; parada após 5 épocas sem melhora |
| Taxa inicial | 0,00003 com AdamW |
| Escolha do checkpoint | mAP50–95 na Home VAL |
| Classes sem cobertura em OI | Perdas negativas ignoradas; caixas positivas preservadas |
| Misturas de imagens | Mosaic, mixup, cutmix e copy-paste desativados |
| TEST e política V2 | Preservados; TEST fora da seleção e saída de alertas em texto |

O 09 avalia uma integração de dados e treinamento: fonte, amostragem, tratamento de
cobertura e validação mudam em relação ao 07b e serão registrados. Ele reutiliza os
primeiros cinco lotes mesmo que você amplie o download do 08 para 100 mil imagens.


## 1. Instalar a versão verificada

Selecione `Ambiente de execução → Alterar tipo de ambiente de execução → Python 3 → GPU T4`.
Se o Colab solicitar reinício após instalar as dependências, reinicie e continue nas próximas células.


In [1]:
%pip install -q ultralytics==8.4.150 pyyaml pandas pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 6.1 MB/s eta 0:00:00


## 2. Carregar a preparação e o treinador com cobertura parcial

Execute esta célula. Ela contém todo o código necessário; não é preciso enviar arquivos
Python adicionais. O modelo permanece um DetectionModel padrão, e a adaptação da perda
é utilizada somente durante o treinamento.


In [2]:
import os
os.environ["YOLO_OFFLINE"] = "true"  # todos os dados e pesos necessários já estarão no Drive


In [3]:
"""Experimento 09: caixas humanas parciais, fontes equilibradas e Home VAL."""
from __future__ import annotations

import hashlib
import inspect
import io
import json
import math
import shutil
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import ultralytics
import yaml
from PIL import Image
from torch.utils.data import DataLoader, Sampler
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils.torch_utils import unwrap_model


VERSION = "09-openimages-parcial-1.0"
ULTRALYTICS_VERSION = "8.4.150"
LOSS_SOURCE_SHA = "357053aa9a24ad2c4d541c3ada3a27232a10d679f89c7b4957b9a84a7da4ec1d"

NOMES = [
    "bed", "sofa", "chair", "table", "lamp", "tv",
    "laptop", "wardrobe", "window", "door",
    "potted plant", "photo frame"
]

ALL_KNOWN = (1 << 12) - 1


def sha256_file(p):
    h = hashlib.sha256()

    with Path(p).open("rb") as f:
        for b in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(b)

    return h.hexdigest()


def canonical_json(x):
    return json.dumps(
        x,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )


def save_json(p, x):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)

    tmp = p.with_name(p.name + ".tmp")
    tmp.write_text(
        json.dumps(x, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    tmp.replace(p)


def check_api():
    if ultralytics.__version__ != ULTRALYTICS_VERSION:
        raise RuntimeError(
            f"Instale ultralytics=={ULTRALYTICS_VERSION} "
            "e reinicie o ambiente se necessário."
        )

    if sha256_file(inspect.getfile(v8DetectionLoss)) != LOSS_SOURCE_SHA:
        raise RuntimeError(
            "A implementação da perda difere da versão verificada; "
            "interrompido antes do treino."
        )


class BCEComCobertura(nn.Module):
    """Zera perdas negativas de classes desconhecidas; conserva positivos."""

    def __init__(self, original=None):
        super().__init__()
        self.original = original or nn.BCEWithLogitsLoss(reduction="none")
        self.known = None

    def forward(self, logits, target):
        if self.known is None:
            raise RuntimeError(
                "Cobertura não foi transmitida à perda parcial."
            )

        known = self.known.to(
            device=logits.device,
            dtype=torch.bool
        )

        if known.shape != (logits.shape[0], logits.shape[-1]):
            raise RuntimeError(
                "Dimensões de cobertura incompatíveis com batch/classes."
            )

        permitted = known[:, None, :] | (target > 0)

        return self.original(logits, target) * permitted.to(logits.dtype)


class DetectionLossParcial(v8DetectionLoss):
    """Altera somente a BCE; caixa, DFL, assignment e ganhos são os padrões."""

    def __init__(self, model):
        super().__init__(model)

        if (
            self.nc != 12
            or not isinstance(self.bce, nn.BCEWithLogitsLoss)
            or self.bce.reduction != "none"
        ):
            raise RuntimeError(
                "Cabeça ou perda de detecção não é a esperada "
                "para este experimento."
            )

        self.bce = BCEComCobertura(self.bce)

    def loss(self, preds, batch):
        if "class_known" not in batch:
            raise RuntimeError(
                "Batch de treinamento sem cobertura; "
                "não substituir por negativos presumidos."
            )

        self.bce.known = batch["class_known"]

        try:
            return super().loss(preds, batch)
        finally:
            self.bce.known = None


class SamplerDuasFontes(Sampler):
    """Cada batch contém metade Home e metade OI; uma época cobre todos os OI."""

    def __init__(self, home_indices, oi_indices, batch_size, seed=42):
        self.home = list(home_indices)
        self.oi = list(oi_indices)

        if (
            not self.home
            or not self.oi
            or batch_size < 2
            or batch_size % 2
        ):
            raise ValueError(
                "São necessárias duas fontes e um batch par >=2."
            )

        self.batch = int(batch_size)
        self.half = self.batch // 2
        self.batches = math.ceil(len(self.oi) / self.half)
        self.seed = int(seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return self.batches * self.batch

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)

        home, oi = [], []
        needed = self.batches * self.half

        while len(home) < needed:
            home.extend(rng.permutation(self.home).tolist())

        while len(oi) < needed:
            oi.extend(rng.permutation(self.oi).tolist())

        for start in range(0, needed, self.half):
            batch = (
                home[start:start + self.half]
                + oi[start:start + self.half]
            )
            rng.shuffle(batch)
            yield from batch


class LoaderFinito(DataLoader):
    def reset(self):
        # Um novo iterador é criado em cada época.
        pass


def epoch_sampler(trainer):
    sampler = getattr(trainer.train_loader, "sampler", None)

    if isinstance(sampler, SamplerDuasFontes):
        sampler.set_epoch(trainer.epoch)


class TrainerOpenImages(DetectionTrainer):
    # Configurado antes de criar o trainer.
    # Não é anexado ao DetectionModel salvo.
    source_records = None

    def __init__(self, *args, **kwargs):
        check_api()
        super().__init__(*args, **kwargs)
        self.add_callback("on_train_epoch_start", epoch_sampler)

    def get_dataloader(
        self,
        dataset_path,
        batch_size=16,
        rank=0,
        mode="train"
    ):
        if mode != "train":
            return super().get_dataloader(
                dataset_path,
                batch_size,
                rank,
                mode
            )

        if self.source_records is None:
            raise RuntimeError(
                "As fontes/coberturas ainda não foram configuradas."
            )

        if self.world_size > 1 or self.args.compile or self.args.rect:
            raise RuntimeError(
                "Este experimento usa uma GPU, "
                "compile=False e rect=False."
            )

        for k in ["mosaic", "mixup", "cutmix", "copy_paste"]:
            if float(getattr(self.args, k, 0)) != 0:
                raise RuntimeError(
                    f"{k} deve ficar em zero para preservar "
                    "a cobertura por imagem."
                )

        dataset = self.build_dataset(
            dataset_path,
            mode,
            batch_size
        )

        home, oi = [], []

        for i, arquivo in enumerate(dataset.im_files):
            key = Path(arquivo).resolve().as_posix()

            if key not in self.source_records:
                raise RuntimeError(
                    f"Imagem sem proveniência/cobertura: {arquivo}"
                )

            source = self.source_records[key]["source"]

            if source == "home":
                home.append(i)
            elif source == "openimages":
                oi.append(i)
            else:
                raise RuntimeError(
                    "Fonte de treinamento não reconhecida."
                )

        sampler = SamplerDuasFontes(
            home,
            oi,
            batch_size,
            self.args.seed
        )

        generator = torch.Generator().manual_seed(self.args.seed)

        return LoaderFinito(
            dataset,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=self.args.workers,
            pin_memory=self.device.type == "cuda",
            collate_fn=dataset.collate_fn,
            generator=generator,
            drop_last=False
        )

    def preprocess_batch(self, batch):
        batch = super().preprocess_batch(batch)

        masks = []

        for arquivo in batch["im_file"]:
            key = Path(arquivo).resolve().as_posix()

            if key not in self.source_records:
                raise RuntimeError(
                    "Batch contém imagem fora do manifesto de treino."
                )

            bitmask = self.source_records[key]["known_mask"]

            masks.append([
                bool(bitmask & (1 << cid))
                for cid in range(12)
            ])

        batch["class_known"] = torch.tensor(
            masks,
            dtype=torch.bool,
            device=self.device
        )

        model = unwrap_model(self.model)
        criterion = getattr(model, "criterion", None)

        if criterion is None:
            # Modelo padrão, já colocado no dispositivo.
            # A EMA foi criada antes desta etapa e valida Home
            # com a perda padrão de cobertura completa.
            model.criterion = DetectionLossParcial(model)

        elif not isinstance(criterion, DetectionLossParcial):
            raise RuntimeError(
                "O treino iniciou com uma perda diferente "
                "da perda parcial."
            )

        return batch


def self_test_bce():
    logits = torch.zeros(
        (2, 3, 12),
        requires_grad=True
    )

    targets = torch.zeros_like(logits)
    targets[1, 0, 7] = 1

    known = torch.ones(
        (2, 12),
        dtype=torch.bool
    )
    known[1, 7] = False
    known[1, 10] = False

    partial = BCEComCobertura()
    partial.known = known
    partial(logits, targets).sum().backward()

    assert logits.grad[1, :, 10].eq(0).all()
    assert logits.grad[1, 1:, 7].eq(0).all()
    assert logits.grad[1, 0, 7].item() < 0
    assert logits.grad[0].ne(0).all()

    full = BCEComCobertura()
    full.known = torch.ones(
        (2, 12),
        dtype=torch.bool
    )

    torch.testing.assert_close(
        full(logits.detach(), targets),
        nn.BCEWithLogitsLoss(reduction="none")(
            logits.detach(),
            targets
        )
    )

    return {
        "gradiente_negativo_desconhecido_zero": True,
        "caixa_positiva_preservada": True,
        "cobertura_completa_igual_bce_padrao": True
    }


def read_safe(z, name):
    path = PurePosixPath(name)

    if path.is_absolute() or ".." in path.parts or "\\" in name:
        raise RuntimeError(
            f"Caminho inseguro dentro de pacote: {name}"
        )

    return z.read(name)


def validate_yolo(raw, nclasses=12, source="<anotação>"):
    """
    Ignora linhas vazias e valida as linhas preenchidas.
    Em caso de erro, informa arquivo, linha e conteúdo.
    """
    result = []

    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError as exc:
        raise RuntimeError(
            f"Anotação YOLO não está em UTF-8: {source}"
        ) from exc

    for line_number, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue

        values = line.split()
        location = f"{source}, linha {line_number}"

        if len(values) != 5:
            raise RuntimeError(
                f"Anotação YOLO inválida em {location}: "
                "esperados 5 campos "
                "(classe x y largura altura), "
                f"encontrados {len(values)}. "
                f"Conteúdo: {line[:160]!r}"
            )

        try:
            cid, *bbox = map(float, values)
        except ValueError as exc:
            raise RuntimeError(
                f"Campo não numérico em {location}: "
                f"{line[:160]!r}"
            ) from exc

        if (
            not cid.is_integer()
            or not 0 <= cid < nclasses
            or not all(math.isfinite(x) for x in bbox)
        ):
            raise RuntimeError(
                f"Classe ou caixa YOLO inválida em {location}: "
                f"{line[:160]!r}"
            )

        x, y, w, h = bbox

        if not (
            0 <= x <= 1
            and 0 <= y <= 1
            and 0 < w <= 1
            and 0 < h <= 1
            and x - w / 2 >= -1e-6
            and x + w / 2 <= 1 + 1e-6
            and y - h / 2 >= -1e-6
            and y + h / 2 <= 1 + 1e-6
        ):
            raise RuntimeError(
                f"Caixa YOLO fora dos limites em {location}: "
                f"{line[:160]!r}"
            )

        result.append(int(cid))

    if not result:
        raise RuntimeError(
            f"Imagem desta preparação sem caixas: {source}"
        )

    return result


def write_checked(dest, raw, expected_sha):
    if hashlib.sha256(raw).hexdigest() != expected_sha:
        raise RuntimeError(
            "Conteúdo do pacote diferente do SHA-256 no manifesto."
        )

    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(raw)


def prepare_training(
    zip_home,
    oi_drive,
    local,
    results,
    expected,
    batches_oi=5,
    home_train_count=1824,
    home_val_count=456,
    oi_count=5000
):
    """Snapshot local somente Home TRAIN/VAL e dos primeiros cinco lotes OI."""

    local = Path(local)
    results = Path(results)
    oi_drive = Path(oi_drive)

    local.mkdir(parents=True, exist_ok=True)
    results.mkdir(parents=True, exist_ok=True)

    prep_cfg = json.loads(
        (oi_drive / "configuracao_preparacao.json").read_text()
    )

    cfg_sha = hashlib.sha256(
        canonical_json(prep_cfg).encode()
    ).hexdigest()

    if (
        cfg_sha != expected["oi_config_sha256"]
        or prep_cfg["nomes"] != NOMES
    ):
        raise RuntimeError(
            "Preparação Open Images diferente da auditada no piloto."
        )

    home_zip_local = local / "home_sun_snapshot.zip"
    shutil.copyfile(zip_home, home_zip_local)

    if sha256_file(home_zip_local) != expected["home_zip_sha256"]:
        raise RuntimeError(
            "ZIP Home+SUN diferente do preservado no 07b/08."
        )

    records, home_val = {}, []
    extract_home = local / "base_existente"

    with zipfile.ZipFile(home_zip_local) as z:
        if len(z.namelist()) != len(set(z.namelist())):
            raise RuntimeError(
                "ZIP Home+SUN tem nomes repetidos."
            )

        manifest = json.loads(
            read_safe(z, "verificacao_preparacao.json")
        )

        if (
            manifest["test"]["imagens"] != 403
            or manifest["test"]["sha256"] != expected["test_sha256"]
        ):
            raise RuntimeError(
                "TEST preservado diferente do esperado."
            )

        if (
            {int(k): v for k, v in manifest["classes"].items()}
            != dict(enumerate(NOMES))
        ):
            raise RuntimeError(
                "Classes/IDs da base Home incompatíveis."
            )

        root = manifest["dataset_dir"]

        for item in manifest["arquivos"]:
            name = item["arquivo"]
            p = PurePosixPath(name)

            in_home_train = (
                name.startswith(f"{root}/images/train/raw_train/")
                or name.startswith(f"{root}/labels/train/raw_train/")
            )

            in_home_val = (
                name.startswith(f"{root}/images/val/raw_train/")
                or name.startswith(f"{root}/labels/val/raw_train/")
            )

            code_or_meta = (
                name.startswith("codigo/")
                or (
                    p.suffix in {
                        ".json", ".csv", ".yaml", ".yml", ".txt"
                    }
                    and "/images/" not in name
                    and "/labels/" not in name
                )
            )

            if not (in_home_train or in_home_val or code_or_meta):
                continue

            raw = read_safe(z, name)
            dest = extract_home / name
            write_checked(dest, raw, item["sha256"])

            if name.startswith(f"{root}/images/train/raw_train/"):
                records[dest.resolve().as_posix()] = {
                    "source": "home",
                    "known_mask": ALL_KNOWN,
                    "sha256": item["sha256"]
                }

            elif name.startswith(f"{root}/images/val/raw_train/"):
                home_val.append(
                    dest.resolve().as_posix()
                )

        listed = {
            item["arquivo"]
            for item in manifest["arquivos"]
        }

        # O hash do ZIP também autentica o código auxiliar
        # que não esteja listado individualmente no manifesto.
        for name in set(z.namelist()) - listed:
            if name.startswith("codigo/") and not name.endswith("/"):
                raw = read_safe(z, name)

                write_checked(
                    extract_home / name,
                    raw,
                    hashlib.sha256(raw).hexdigest()
                )

    home_dataset = extract_home / root
    home_images = list(records)

    if (
        len(home_images) != home_train_count
        or len(home_val) != home_val_count
    ):
        raise RuntimeError(
            "Home TRAIN/VAL inesperados: "
            f"{len(home_images)}/{len(home_val)}; "
            f"esperados {home_train_count}/{home_val_count}."
        )

    for image in [*home_images, *home_val]:
        label = Path(
            image.replace("/images/", "/labels/")
        ).with_suffix(".txt")

        validate_yolo(
            label.read_bytes(),
            source=str(label)
        )

    packages = json.loads(
        (oi_drive / "manifesto_pacotes.json").read_text()
    )

    if packages["config_sha256"] != cfg_sha:
        raise RuntimeError(
            "Manifesto dos lotes tem configuração diferente."
        )

    selected = sorted(
        packages["lotes"],
        key=lambda r: r["arquivo"]
    )[:batches_oi]

    if len(selected) != batches_oi:
        raise RuntimeError(
            "Não há cinco lotes confirmados do piloto."
        )

    oi_ids, oi_hashes, package_snapshot = set(), set(), []
    oi_root = local / "openimages"
    per_class = Counter()

    for item in selected:
        name = item["arquivo"]

        if PurePosixPath(name).name != name:
            raise RuntimeError(
                "Nome de pacote não é um nome simples."
            )

        snapshot = local / name

        shutil.copyfile(
            oi_drive / "lotes" / name,
            snapshot
        )

        if (
            sha256_file(snapshot) != item["zip_sha256"]
            or snapshot.stat().st_size != item["zip_bytes"]
        ):
            raise RuntimeError(
                f"Pacote alterado ou incompleto: {name}"
            )

        with zipfile.ZipFile(snapshot) as z:
            if len(z.namelist()) != len(set(z.namelist())):
                raise RuntimeError(
                    "ZIP Open Images tem membros repetidos."
                )

            m = json.loads(
                read_safe(z, "manifesto_lote.json")
            )

            if (
                m["config_sha256"] != cfg_sha
                or m["origem_split"] != "train"
            ):
                raise RuntimeError(
                    "Lote não é da seleção do treino "
                    "Open Images auditado."
                )

            for r in m["aceitas"]:
                if r["id"] in oi_ids or r["sha256"] in oi_hashes:
                    raise RuntimeError(
                        "ID ou bytes duplicados nos lotes usados no treino."
                    )

                oi_ids.add(r["id"])
                oi_hashes.add(r["sha256"])

                im_raw = read_safe(z, r["imagem"])
                label_raw = read_safe(z, r["label"])
                cov_raw = read_safe(z, r["cobertura"])

                for key, raw, hkey in [
                    ("imagem", im_raw, "sha256"),
                    ("label", label_raw, "label_sha256"),
                    ("cobertura", cov_raw, "cobertura_sha256")
                ]:
                    write_checked(
                        oi_root / r[key],
                        raw,
                        r[hkey]
                    )

                cls = validate_yolo(
                    label_raw,
                    source=f"{name}:{r['label']}"
                )

                cov = json.loads(cov_raw)

                if (
                    cov["known_mask"] != r["known_mask"]
                    or not isinstance(r["known_mask"], int)
                    or not 0 <= r["known_mask"] <= ALL_KNOWN
                ):
                    raise RuntimeError(
                        "Cobertura do lote inconsistente."
                    )

                known_list = (
                    set(cov["classes_positivas_verificadas_exaustivas"])
                    | set(cov["classes_negativas_verificadas"])
                )

                if sum(1 << cid for cid in known_list) != r["known_mask"]:
                    raise RuntimeError(
                        "Lista de classes conhecidas "
                        "não corresponde à máscara."
                    )

                if set(cls) & set(cov["classes_negativas_verificadas"]):
                    raise RuntimeError(
                        "Caixa positiva contradiz ausência verificada."
                    )

                with Image.open(io.BytesIO(im_raw)) as im:
                    im.load()

                    if im.size != (r["width"], r["height"]):
                        raise RuntimeError(
                            "Dimensões da imagem diferem do manifesto."
                        )

                image = (
                    oi_root / r["imagem"]
                ).resolve().as_posix()

                records[image] = {
                    "source": "openimages",
                    "known_mask": r["known_mask"],
                    "sha256": r["sha256"]
                }

                per_class.update(cls)

        package_snapshot.append(item)

        # Remove apenas a cópia local do ZIP do lote.
        snapshot.unlink()

    if len(oi_ids) != oi_count:
        raise RuntimeError(
            f"Pilotagem esperava {oi_count} imagens OI, "
            f"encontrou {len(oi_ids)}."
        )

    if set(home_val) & set(records):
        raise RuntimeError(
            "Imagem de validação aparece no treino."
        )

    val_hashes = {
        sha256_file(x)
        for x in home_val
    }

    if val_hashes & {
        r["sha256"]
        for r in records.values()
    }:
        raise RuntimeError(
            "Imagem de treino coincide em bytes "
            "com a validação Home."
        )

    # O 08 já realizou a triagem de contaminação.
    # Aqui repetimos a verificação por bytes contra Home VAL.
    train_list = local / "train.txt"
    val_list = local / "home_val.txt"

    train_list.write_text(
        "\n".join(sorted(records)) + "\n",
        encoding="utf-8"
    )

    val_list.write_text(
        "\n".join(sorted(home_val)) + "\n",
        encoding="utf-8"
    )

    data_yaml = local / "data_home_openimages.yaml"

    data_yaml.write_text(
        yaml.safe_dump(
            {
                "path": str(local.resolve()),
                "train": str(train_list),
                "val": str(val_list),
                "names": dict(enumerate(NOMES))
            },
            sort_keys=False,
            allow_unicode=True
        ),
        encoding="utf-8"
    )

    save_json(
        local / "fontes_cobertura_treino.json",
        records
    )

    summary = {
        "versao": VERSION,
        "home_train_imagens_unicas": len(home_images),
        "home_val_imagens": len(home_val),
        "openimages_imagens_unicas": len(oi_ids),
        "treino_imagens_unicas": len(records),
        "caixas_openimages": sum(per_class.values()),
        "caixas_openimages_por_classe": dict(per_class),
        "lotes_utilizados": package_snapshot,
        "home_zip_sha256": expected["home_zip_sha256"],
        "test_sha256": expected["test_sha256"],
        "test_imagens": 403,
        "test_extraido_ou_avaliado": False,
        "sun_incluida_no_novo_treino": False,
        "amostragem": (
            "50% Home / 50% OI em cada batch; "
            "Home reamostrado por índices, sem duplicar arquivos"
        ),
        "selecao_checkpoint": (
            "mAP50-95 Home VAL; triagem de alertas após treino"
        ),
        "tratamento_parcial": (
            "Zero gradiente BCE para negativos de classes desconhecidas; "
            "caixas positivas preservadas"
        ),
        "config_preparacao_sha256": cfg_sha
    }

    save_json(
        results / "resumo_dados_treino.json",
        summary
    )

    save_json(
        results / "lotes_utilizados.json",
        package_snapshot
    )

    TrainerOpenImages.source_records = records

    return {
        "yaml": data_yaml,
        "home_dataset": home_dataset,
        "codigo": extract_home / "codigo",
        "records": records,
        "home_val": home_val,
        "resumo": summary
    }


def export_standard_checkpoint(source, dest):
    """Confere que o checkpoint não depende das classes do treinador parcial."""

    obj = torch.load(
        source,
        map_location="cpu",
        weights_only=False
    )

    model = obj.get("ema") or obj.get("model")

    if type(model) is not DetectionModel:
        raise RuntimeError(
            "O checkpoint não contém DetectionModel padrão."
        )

    model.criterion = None
    model = model.float().eval()

    if not all(
        torch.isfinite(t).all()
        for t in model.state_dict().values()
        if t.is_floating_point()
    ):
        raise RuntimeError(
            "Checkpoint com pesos não finitos."
        )

    obj["model"] = model
    obj["ema"] = None
    obj["optimizer"] = None

    torch.save(obj, dest)

    return {
        "arquivo": str(dest),
        "sha256": sha256_file(dest),
        "modelo_padrao": True,
        "criterion_serializada": False
    }

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
from ultralytics import YOLO, settings
from datetime import datetime, timezone
import platform
import sys

settings.update({k: False for k in ["sync", "hub", "platform", "wandb", "comet", "mlflow",
                                   "neptune", "clearml", "dvc", "tensorboard"] if k in settings})
check_api()
if not torch.cuda.is_available():
    raise RuntimeError("Ative a GPU T4 e reconecte o ambiente antes de treinar.")
TESTE_PERDA = self_test_bce()
print("GPU:", torch.cuda.get_device_name(0))
print("Verificação da perda:", TESTE_PERDA)


GPU: Tesla T4
Verificação da perda: {'gradiente_negativo_desconhecido_zero': True, 'caixa_positiva_preservada': True, 'cobertura_completa_igual_bce_padrao': True}


## 3. Montar o Drive e localizar as entradas

Reutilize os arquivos já disponíveis:

```text
Meu Drive/TCC_Visao_Computacional/
  datasets/homeobjects_sunrgbd_validado.zip
  datasets/openimages_v7_tcc_100k_v1/configuracao_preparacao.json
  datasets/openimages_v7_tcc_100k_v1/manifesto_pacotes.json
  datasets/openimages_v7_tcc_100k_v1/lotes/lote_000001.zip ... lote_000005.zip
  modelos/yolov8n_baseline/best.pt
```

A extração será no disco local da GPU. Não treinamos lendo milhares de JPEGs diretamente
do Drive. Uma pasta nova de resultados será criada para cada execução do 09.


In [5]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_TCC = Path("/content/drive/MyDrive/TCC_Visao_Computacional")
ZIP_HOME = DRIVE_TCC / "datasets/homeobjects_sunrgbd_validado.zip"
PASTA_OI = DRIVE_TCC / "datasets/openimages_v7_tcc_100k_v1"
MODELO_ANTIGO = DRIVE_TCC / "modelos/yolov8n_baseline/best.pt"
SHA_MODELO_ANTIGO = "4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9"
SHA_POLITICA = "1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845"
EXPECTED = {
    "oi_config_sha256": "9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49",
    "home_zip_sha256": "ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676",
    "test_sha256": "f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89",
}
NOME_EXPERIMENTO = "yolov8n_home_openimages_piloto_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
PASTA_LOCAL = Path("/content/tcc_openimages") / NOME_EXPERIMENTO
PASTA_RESULTADOS = DRIVE_TCC / "resultados" / NOME_EXPERIMENTO
EPOCHS = 15
PATIENCE = 5
BATCH = 16
IMGSZ = 640
LR0 = 0.00003
SEED = 42
WORKERS = 2

for p in [ZIP_HOME, MODELO_ANTIGO, PASTA_OI / "manifesto_pacotes.json", PASTA_OI / "configuracao_preparacao.json"]:
    if not p.is_file():
        raise FileNotFoundError(f"Entrada ausente: {p}")
if sha256_file(MODELO_ANTIGO) != SHA_MODELO_ANTIGO:
    raise RuntimeError("Os pesos iniciais não são o modelo antigo avaliado.")
if BATCH < 2 or BATCH % 2:
    raise ValueError("O batch precisa ser par para a composição 50/50.")
PASTA_LOCAL.mkdir(parents=True, exist_ok=False)
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=False)
if shutil.disk_usage(PASTA_LOCAL).free < 10 * 1024**3:
    raise RuntimeError("Reserve pelo menos 10 GiB no armazenamento temporário do Colab.")
MODELO_INICIAL = PASTA_LOCAL / "baseline_inicial.pt"
shutil.copyfile(MODELO_ANTIGO, MODELO_INICIAL)
print("Resultados:", PASTA_RESULTADOS)


Mounted at /content/drive
Resultados: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_home_openimages_piloto_20261014_135030_882898


## 4. Montar e conferir o snapshot do piloto

Verificamos hashes dos ZIPs, imagens, caixas e coberturas. O treino inclui apenas Home
TRAIN e os primeiros cinco lotes de OI; SUN não entra neste experimento. O TEST é conferido
pela referência e pelo SHA do pacote preservado, sem extrair suas imagens ou avaliá-las.

Os exemplos Home são reamostrados por índices para equilibrar os batches. Isso não cria
novas imagens nem duplica arquivos: há 6.824 imagens únicas e 10.000 apresentações por época.


In [6]:
DADOS = prepare_training(ZIP_HOME, PASTA_OI, PASTA_LOCAL, PASTA_RESULTADOS, EXPECTED)
YAML_TREINO = DADOS["yaml"]
NOMES_DICT = dict(enumerate(NOMES))
sys.path.insert(0, str(DADOS["codigo"]))
from src.validacao_alertas import carregar_home_val, avaliar_alertas
import src.politica_alertas as politica
if sha256_file(politica.__file__) != SHA_POLITICA:
    raise RuntimeError("Política V2 diferente da utilizada nas avaliações anteriores.")
REGISTROS_HOME_VAL = carregar_home_val(DADOS["home_dataset"], NOMES_DICT)
if len(REGISTROS_HOME_VAL) != 456:
    raise RuntimeError("Validação Home não corresponde às 456 imagens preservadas.")
sampler_info = SamplerDuasFontes(range(1824), range(1824, 6824), BATCH, SEED)
CONFIGURACAO = {
    "versao": VERSION, "experimento": NOME_EXPERIMENTO,
    "modelo_base_sha256": SHA_MODELO_ANTIGO, **EXPECTED,
    "home_train_imagens_unicas": 1824, "openimages_imagens_unicas": 5000,
    "home_val_imagens": 456, "apresentacoes_por_epoca": len(sampler_info),
    "batches_por_epoca": len(sampler_info) // BATCH,
    "proporcao_batch": {"home": 0.5, "openimages": 0.5},
    "epochs": EPOCHS, "patience": PATIENCE, "batch": BATCH, "imgsz": IMGSZ,
    "lr0": LR0, "lrf": 0.1, "optimizer": "AdamW", "cos_lr": True,
    "warmup_epochs": 2.0, "warmup_bias_lr": 0.0, "weight_decay": 0.0005,
    "seed": SEED, "workers": WORKERS, "resume": False,
    "mosaic": 0.0, "mixup": 0.0, "cutmix": 0.0, "copy_paste": 0.0,
    "close_mosaic": 0, "scale": 0.2, "translate": 0.1, "fliplr": 0.5,
    "compile": False, "cls_pw": 0.0,
    "criterio_checkpoint": "best.pt por fitness de Home VAL no Ultralytics (mAP50-95)",
    "criterio_alertas": "Precisão maior, recall preservado e omissões de alta prioridade não maiores",
    "politica_sha256": SHA_POLITICA, "test_avaliado_para_selecao": False,
    "perda_parcial": "BCE negativa desconhecida mascarada; positivos preservados; box/DFL padrão",
    "loss_source_sha256": LOSS_SOURCE_SHA, "teste_perda": TESTE_PERDA,
    "python": platform.python_version(), "ultralytics": ultralytics.__version__,
    "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
}
save_json(PASTA_RESULTADOS / "configuracao_experimento.json", CONFIGURACAO)
print(json.dumps(DADOS["resumo"], ensure_ascii=False, indent=2))
print("Apresentações/época:", len(sampler_info), "| batches/época:", len(sampler_info)//BATCH)


{
  "versao": "09-openimages-parcial-1.0",
  "home_train_imagens_unicas": 1824,
  "home_val_imagens": 456,
  "openimages_imagens_unicas": 5000,
  "treino_imagens_unicas": 6824,
  "caixas_openimages": 20341,
  "caixas_openimages_por_classe": {
    "0": 795,
    "7": 193,
    "8": 3949,
    "10": 2625,
    "2": 4603,
    "9": 953,
    "1": 1121,
    "3": 3272,
    "4": 657,
    "5": 441,
    "6": 563,
    "11": 1169
  },
  "lotes_utilizados": [
    {
      "arquivo": "lote_000001.zip",
      "zip_sha256": "9be878dd7c8f91dba90deb1cea226fc28bd6e3c5ad6192ced1a0ee7e8369c949",
      "zip_bytes": 287039437
    },
    {
      "arquivo": "lote_000002.zip",
      "zip_sha256": "e9032df3a116eddc792d645885034dc4108636f18f8636a49fd61c89d6d8cfdd",
      "zip_bytes": 287377350
    },
    {
      "arquivo": "lote_000003.zip",
      "zip_sha256": "6a0d9a96647c8bc200f752a4e3d825bc371d2a6cc4d792fb779d908bcd59f0f7",
      "zip_bytes": 291260618
    },
    {
      "arquivo": "lote_000004.zip",
      "zip_sh

## 5. Avaliar novamente o antigo nas mesmas imagens Home VAL

O controle e o novo candidato usam a mesma validação e os mesmos parâmetros. Os alertas
são comparados com a V2 fixa aplicada às anotações, mantendo a referência dos notebooks
anteriores. Isso não é uma validação independente de perigo físico.


In [7]:
def avaliar_detector_home(modelo, nome):
    r = modelo.val(data=str(YAML_TREINO), split="val", imgsz=IMGSZ, batch=BATCH,
                   device=0, conf=0.001, iou=0.70, max_det=300,
                   plots=True, project=str(PASTA_RESULTADOS), name=nome, exist_ok=False)
    return {"precision_detector":float(r.box.mp), "recall_detector":float(r.box.mr),
            "map50":float(r.box.map50), "map50_95":float(r.box.map)}

antigo = YOLO(str(MODELO_INICIAL))
if {int(k): str(v) for k,v in antigo.names.items()} != NOMES_DICT:
    raise RuntimeError("IDs/classes do modelo antigo diferentes dos dados.")
METRICAS_ANTIGO = {**avaliar_detector_home(antigo, "baseline_home_val"),
                   **avaliar_alertas(antigo, REGISTROS_HOME_VAL, NOMES_DICT)}
save_json(PASTA_RESULTADOS / "metricas_baseline_val.json", METRICAS_ANTIGO)
print(json.dumps(METRICAS_ANTIGO, ensure_ascii=False, indent=2))
del antigo
torch.cuda.empty_cache()


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1760.0±1085.1 MB/s, size: 141.5 KB)
val: Scanning /content/tcc_openimages/yolov8n_home_openimages_piloto_20261014_135030_882898/base_existente/homeobjects_yolo/labels/val/raw_train... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 2.1Kit/s 0.2s
val: New cache created: /content/tcc_openimages/yolov8n_home_openimages_piloto_20261014_135030_882898/base_existente/homeobjects_yolo/labels/val/raw_train.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.9it/s 9.9s
                   all        456       3682      0.883      0.765      0.859      0.664
                   bed         29         29      0.907      0.793      0.879      0.719
                  sofa        294        420      0.912   

## 6. Treinar o detector

Iniciamos novamente nos pesos antigos. Cada batch tem metade Home e metade Open Images.
Os objetos anotados continuam supervisionados; classes sem cobertura deixam de produzir
perdas negativas nas respectivas imagens OI. Os aumentos que misturam fotos ficam desativados.

O limite de 15 épocas considera que cada época tem 10 mil apresentações. A validação Home
escolhe o `best.pt` e controla a parada após cinco épocas sem melhora. O modelo antigo
permanece guardado; esta execução cria um candidato em uma pasta própria.

Checkpoints de cada época ficam no Drive. Em caso de interrupção da sessão, preservar a pasta e consultar
o notebook/log: a retomada exige conservar este treinador e o snapshot, não abrir o peso
com um treino padrão sem cobertura. Uma reexecução completa desta versão cria um experimento novo.


In [8]:
modelo = YOLO(str(MODELO_INICIAL))
resultado_treino = modelo.train(
    trainer=TrainerOpenImages,
    data=str(YAML_TREINO), epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
    imgsz=IMGSZ, device=0, seed=SEED, workers=WORKERS,
    optimizer="AdamW", lr0=LR0, lrf=0.10, cos_lr=True,
    warmup_epochs=2.0, warmup_bias_lr=0.0, weight_decay=0.0005,
    mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0, close_mosaic=0,
    scale=0.2, translate=0.1, fliplr=0.5,
    compile=False, rect=False, multi_scale=0.0, cls_pw=0.0,
    resume=False, pretrained=True, cache=False, max_det=300,
    save=True, save_period=1, plots=True,
    project=str(PASTA_RESULTADOS), name="treinamento", exist_ok=False,
)
BEST_RUN = Path(modelo.trainer.best)
if not BEST_RUN.is_file():
    raise RuntimeError("Treino não produziu best.pt.")
EXPORT = export_standard_checkpoint(BEST_RUN, PASTA_RESULTADOS / "best.pt")
save_json(PASTA_RESULTADOS / "exportacao_checkpoint.json", EXPORT)
print("Candidato exportado em formato padrão:", EXPORT["arquivo"])


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/tcc_openimages/yolov8n_home_openimages_piloto_20261014_135030_882898/data_home_openimages.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=3e-05, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/tcc_openimages/yolov8n_home

## 7. Comparar o candidato com o antigo

Mantemos os parâmetros de avaliação e a política V2. O checkpoint é selecionado em Home VAL;
TEST continua fora da escolha. O candidato só será considerado promissor se aumentar a
precisão dos alertas, preservar o recall e não aumentar omissões de alta prioridade.


In [9]:
candidato = YOLO(str(PASTA_RESULTADOS / "best.pt"))
METRICAS_CANDIDATO = {**avaliar_detector_home(candidato, "candidato_home_val"),
                      **avaliar_alertas(candidato, REGISTROS_HOME_VAL, NOMES_DICT)}
save_json(PASTA_RESULTADOS / "metricas_candidato_val.json", METRICAS_CANDIDATO)
CHAVES = ["precision_detector", "recall_detector", "map50", "map50_95",
          "precision_alertas", "recall_alertas", "f1_alertas", "fn", "fp", "alta_sem_alerta"]
comparacao = pd.DataFrame([
    {"metrica":k, "antigo":METRICAS_ANTIGO[k], "candidato":METRICAS_CANDIDATO[k],
     "diferenca":None if METRICAS_ANTIGO[k] is None or METRICAS_CANDIDATO[k] is None
                    else METRICAS_CANDIDATO[k] - METRICAS_ANTIGO[k]}
    for k in CHAVES
])
comparacao.to_csv(PASTA_RESULTADOS / "comparacao_validacao.csv", index=False, encoding="utf-8-sig")
display(comparacao)
pa, pn = METRICAS_ANTIGO["precision_alertas"], METRICAS_CANDIDATO["precision_alertas"]
ra, rn = METRICAS_ANTIGO["recall_alertas"], METRICAS_CANDIDATO["recall_alertas"]
promissor = (all(v is not None for v in [pa,pn,ra,rn]) and pn > pa and rn >= ra
             and METRICAS_CANDIDATO["alta_sem_alerta"] <= METRICAS_ANTIGO["alta_sem_alerta"])
TRIAGEM = {"promissor_na_validacao":bool(promissor),
    "criterio":"Precisão de alertas maior, recall preservado e omissões de alta prioridade não maiores",
    "observacao":"Triagem exploratória; baseline pode ter visto parte de Home VAL. Referência V2 não valida risco físico independentemente.",
    "proximo_passo":"Se promissor, planejar avaliação final preservada. Se não, manter antigo e revisar dados/treino antes de ampliar."}
save_json(PASTA_RESULTADOS / "triagem_validacao.json", TRIAGEM)
print(json.dumps(TRIAGEM, ensure_ascii=False, indent=2))
print("Arquivos para enviar:", PASTA_RESULTADOS)


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2374.2±720.4 MB/s, size: 163.5 KB)
val: Scanning /content/tcc_openimages/yolov8n_home_openimages_piloto_20261014_135030_882898/base_existente/homeobjects_yolo/labels/val/raw_train.cache... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 147.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 3.5it/s 8.4s
                   all        456       3682      0.848      0.778      0.864      0.665
                   bed         29         29      0.808      0.793      0.876      0.728
                  sofa        294        420       0.93      0.914      0.957      0.786
                 chair        200        428      0.866      0.815      0.901      0.693
                 table        331 

,metrica,antigo,candidato,diferenca
0,precision_detector,0.882927,0.847865,-0.035062
1,recall_detector,0.765052,0.778281,0.013229
2,map50,0.859207,0.863786,0.004578
3,map50_95,0.664002,0.664595,0.000592
4,precision_alertas,0.783646,0.798256,0.014610
5,recall_alertas,0.899023,0.894463,-0.004560
6,f1_alertas,0.837379,0.843625,0.006247
7,fn,155.000000,162.000000,7.000000
8,fp,381.000000,347.000000,-34.000000
9,alta_sem_alerta,24.000000,23.000000,-1.000000


{
  "promissor_na_validacao": false,
  "criterio": "Precisão de alertas maior, recall preservado e omissões de alta prioridade não maiores",
  "observacao": "Triagem exploratória; baseline pode ter visto parte de Home VAL. Referência V2 não valida risco físico independentemente.",
  "proximo_passo": "Se promissor, planejar avaliação final preservada. Se não, manter antigo e revisar dados/treino antes de ampliar."
}
Arquivos para enviar: /content/drive/MyDrive/TCC_Visao_Computacional/resultados/yolov8n_home_openimages_piloto_20261014_135030_882898


## 8. Enviar os resultados e continuar a escala

Arquivos da pasta de resultados desta execução:

- `comparacao_validacao.csv`;
- `configuracao_experimento.json`;
- `triagem_validacao.json`;
- `resumo_dados_treino.json`;
- `treinamento/results.csv`.

Os pesos e os ZIPs são armazenados no Drive. Os registros da célula identificam eventuais falhas.

**Reserva de 100 mil:** o 08 pode continuar com `MAX_LOTES_POR_EXECUCAO=100`, mesma
pasta e meta. A estimativa do piloto foi 27,02 GiB para os ZIPs; confira espaço livre
além do que os outros arquivos já ocupam. A previsão de tempo depende da velocidade
e disponibilidade da próxima sessão. O 09 continua usando os primeiros cinco lotes.

Guarda-roupa tem somente 173 imagens elegíveis neste recorte, e todas já estão no piloto.
O download maior amplia principalmente outras classes. O inventário, a perda parcial,
a proporção de fontes e o resultado em Home VAL orientam o próximo aumento de treino.

## Verificação e referências

Foram testados localmente: zero gradiente para negativos desconhecidos, positivos
preservados, igualdade com a perda padrão para cobertura completa, batches equilibrados,
hashes e separação de fontes, duas épocas sintéticas completas e inferência padrão com
o checkpoint exportado. Os testes foram em CPU/Python 3.12/Torch 2.8; o Colab T4 e os
pesos/dados reais são a execução que você fará. Não há ganho de precisão medido ainda.

- [Open Images: rótulos parciais e hierarquia](https://storage.googleapis.com/openimages/web/evaluation.html)
- [Ultralytics: personalização de treinadores](https://docs.ultralytics.com/usage/engine/)
- [Ultralytics: perda de detecção](https://docs.ultralytics.com/reference/utils/loss/)

A versão 8.4.150 e o hash do arquivo da perda são verificados antes do treinamento.
